# CASCADES v10.3 — Abliterated Qwen3-4B

Continual PEFT on an OBLITERATUS-processed Qwen3-4B using the full CASCADES v10.3 Elastic Riemannian Ecosystem.

**Model:** `Qwen/Qwen3-4B` → abliterated via OBLITERATUS (refusal direction removed)

**Architecture:** `Qwen3ForCausalLM` — 36 layers, 32 Q-heads, 8 KV-heads (GQA 4:1), all full-attention

**CASCADES v10.3 features:**
1. **GQA-Aware Metric Preconditioning** — resolves 8B scaling paradox (auto-detected 4:1 ratio)
2. **Ambient Trace Dedup** — cross-adapter deduplication in sleep
3. **Tikhonov Soft-EAR** — smooth gradient reassignment (strict for frozen, soft for active)
4. **Principal Tangent Expansion** — noise-free rank revival
5. **Subspace-Contrastive Decoding** — adapter-level CFG
6. **v10.3 BWT Fixes** — tangent-first projection, FunLoRA distillation, half-rank plasticity

**Training Data:** Real benchmarks (GSM8K → ARC-Challenge → CommonsenseQA)

**A100 Scaling:** Rank 16 (2x), 512 token context, 4 epochs — adapter weights stay ~15MB for 8GB inference

**Runtime:** ~15–20 min on A100 (40GB)

## 1. Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs.git
%cd CASCADES--continual-PEFT-for-Local-LLMs
!pip install -q -r requirements.txt
!pip install -q peft

## 2. Upload Abliterated Qwen3 Model

Uploads `abliteratedqwen3.zip` (from OBLITERATUS) and extracts it to `abliterated/` so `train.py --model_id ./abliterated` can find it.

**Options (in priority order):**
1. Google Drive — if the zip is at `My Drive/abliteratedqwen3.zip`, it copies automatically
2. Manual upload — a file picker will pop up if Drive copy isn't found

In [ ]:
#@title 2. Upload Abliterated Qwen3 Model { run: "auto" }
import os, zipfile, shutil
from pathlib import Path

MODEL_ZIP = "abliteratedqwen3.zip"
MODEL_DIR = "abliterated"

# Check if already extracted
if os.path.isdir(MODEL_DIR) and any(f.suffix == '.safetensors' for f in Path(MODEL_DIR).iterdir()):
    print(f"✓ {MODEL_DIR}/ already contains safetensors — skipping upload")
else:
    # Mount Google Drive (will prompt for auth)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_path = f"/content/drive/MyDrive/{MODEL_ZIP}"
    except Exception:
        drive_path = None

    if drive_path and os.path.exists(drive_path):
        print(f"Found {MODEL_ZIP} on Google Drive — copying (~6.2 GB)...")
        shutil.copy(drive_path, MODEL_ZIP)
    else:
        print(f"Select {MODEL_ZIP} to upload (or upload it to My Drive and re-run):")
        from google.colab import files
        uploaded = files.upload()
        if MODEL_ZIP not in uploaded:
            raise FileNotFoundError(
                f"Expected '{MODEL_ZIP}', got: {list(uploaded.keys())}\n"
                "Rename your zip to abliteratedqwen3.zip and re-run."
            )

    print(f"Extracting to {MODEL_DIR}/ ...")
    os.makedirs(MODEL_DIR, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP, "r") as zf:
        members = zf.namelist()
        # If zip has one top-level folder, flatten into MODEL_DIR
        top_dirs = {m.split("/")[0] for m in members if "/" in m}
        if len(top_dirs) == 1:
            top = top_dirs.pop()
            for member in members:
                rel = os.path.relpath(member, top)
                if rel == ".":
                    continue
                dest = os.path.join(MODEL_DIR, rel)
                if member.endswith("/"):
                    os.makedirs(dest, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(dest), exist_ok=True)
                    with zf.open(member) as src, open(dest, "wb") as dst:
                        dst.write(src.read())
        else:
            zf.extractall(MODEL_DIR)

    # Clean up zip to free disk
    if os.path.exists(MODEL_ZIP):
        os.remove(MODEL_ZIP)
        print(f"Cleaned up {MODEL_ZIP} to free disk space")

    print(f"✓ Extracted to {MODEL_DIR}/")
    contents = sorted(f.name for f in Path(MODEL_DIR).iterdir())
    print(f"  Contents: {contents}")
    safetensors = [f for f in contents if f.endswith('.safetensors')]
    print(f"  Model shards: {len(safetensors)}")

## 3. Verify GPU & Environment

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")
    if vram < 30:
        print("⚠️  Less than 30GB VRAM detected. Go to Runtime > Change runtime type > A100 GPU")
    else:
        print(f"✓ Sufficient VRAM for Qwen3-4B NF4 + CASCADES rank-16 adapters")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 4. Verify CASCADES v10.3 Library

In [ ]:
import cascades
print(f"CASCADES version: {cascades.__version__}")
assert cascades.__version__ == '2.0.0', f"Expected v2.0.0, got {cascades.__version__}"

# Verify v10.3 features
from cascades.math_ops import gqa_precondition_gradient, soft_ear
from cascades.config import AblationConfig

cfg = AblationConfig()
print(f"GQA ratio:            {cfg.gqa_ratio} (will auto-detect 4:1 from Qwen3-4B)")
print(f"Soft-EAR gamma:       {cfg.ear_gamma}")
print(f"CFG lambda:           {cfg.cfg_lambda}")
print(f"Principal expansion:  {cfg.enable_principal_expansion}")
print(f"Ambient dedup:        {cfg.enable_ambient_dedup}")
print("\n✓ All v10.3 features verified.")

## 5. Verify Abliterated Model Loads

Quick sanity check that the model files are valid and the tokenizer loads.

In [ ]:
import json
from pathlib import Path

# Check model config
config_path = Path("abliterated/config.json")
assert config_path.exists(), "abliterated/config.json not found — re-run step 2"

with open(config_path) as f:
    model_config = json.load(f)

print(f"Architecture: {model_config['architectures'][0]}")
print(f"Hidden size:  {model_config['hidden_size']}")
print(f"Layers:       {model_config['num_hidden_layers']}")
print(f"Q-heads:      {model_config['num_attention_heads']}")
print(f"KV-heads:     {model_config['num_key_value_heads']}")
print(f"GQA ratio:    {model_config['num_attention_heads'] // model_config['num_key_value_heads']}:1")

# Check abliteration metadata
meta_path = Path("abliterated/abliteration_metadata.json")
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"\nOBLITERATUS metadata:")
    print(f"  Source model:  {meta['source_model']}")
    print(f"  Technique:     {meta['technique']}")
    print(f"  Refusal rate:  {meta['quality_metrics']['refusal_rate']:.1%}")
    print(f"  Perplexity:    {meta['quality_metrics']['perplexity']:.2f}")
    print(f"  KL divergence: {meta['quality_metrics']['kl_divergence']:.6f}")
else:
    print("\n(No abliteration metadata found — model will still work)")

# Quick tokenizer load test
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("abliterated")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
test_ids = tokenizer.encode("Hello, world!", return_tensors="pt")
print(f"\n✓ Tokenizer loaded ({tokenizer.__class__.__name__}, vocab={tokenizer.vocab_size})")
print(f"  Test encode: 'Hello, world!' → {test_ids.shape[1]} tokens")

## 6. Download Real Benchmark Data

Downloads 3 real benchmark datasets from HuggingFace:
- **Task 0:** GSM8K (grade-school math, natural CoT)
- **Task 1:** ARC-Challenge (science reasoning)
- **Task 2:** CommonsenseQA (commonsense reasoning)

**300 examples per task** — more diversity reduces task-boundary forgetting and gives the Stiefel manifold a richer null-space to project against.

In [ ]:
!python download_real_data.py --samples 300

## 7. Train CASCADES on Abliterated Qwen3-4B

Runs the full CASCADES v10.3 continual learning pipeline:
- **3 sequential tasks** (GSM8K → ARC → CSQA)
- **NF4 quantization** (bfloat16 compute)
- **Rank 16** Stiefel manifold (A100 has 40GB headroom — 2x rank = richer subspace, more protected directions)
- **512 token context** (vs 384 on 8GB — captures longer chain-of-thought reasoning)
- **4 epochs** per task (A100 throughput makes this fast)
- **D-MoLE** dynamic layer selection
- **Sleep consolidation** between tasks
- **GQA preconditioning** auto-detected at 4:1

**A100 Scaling Philosophy:** Training-time compute scales via rank/epochs/context. The final adapter weights are only ~15MB (even at rank 16), so inference on 8GB VRAM is identical. You're buying a richer Stiefel manifold and better null-space protection.

**For 8GB inference:** `train.py --rank 8 --max_length 384 --epochs 2` (default settings).

In [ ]:
OUTPUT_PREFIX = "cascades_qwen3_abliterated"

!python train.py \
    --model_id ./abliterated \
    --rank 16 \
    --max_length 512 \
    --epochs 4 \
    --lr_riemannian 0.003 \
    --dmole_threshold 0.22 \
    --output_prefix {OUTPUT_PREFIX} \
    --eval_em

## 8. Results — Accuracy Matrix & BWT

In [ ]:
import pandas as pd
import numpy as np

results_csv = f"{OUTPUT_PREFIX}_results.csv"
df = pd.read_csv(results_csv, index_col=0)

print("="*60)
print("CASCADES v10.3 — Abliterated Qwen3-4B Results")
print("="*60)
print("\nAccuracy Matrix (Proxy ACC %):")
print((df * 100).round(2).to_string())

# Backward Transfer
num_tasks = len(df)
bwt_vals = [df.iloc[-1, i] - df.iloc[i, i] for i in range(num_tasks - 1)]
bwt = np.mean(bwt_vals) * 100
avg_acc = df.iloc[-1].mean() * 100

print(f"\nAverage Accuracy: {avg_acc:.2f}%")
print(f"Backward Transfer: {bwt:+.2f}%")
print(f"\nVerdict: {'✓ Zero forgetting confirmed' if bwt >= 0 else '⚠ Some forgetting detected'}")

# Per-task breakdown
print(f"\nPer-task final accuracy:")
task_names = ["GSM8K (Math)", "ARC (Science)", "CSQA (Commonsense)"]
for i, name in enumerate(task_names):
    acc = df.iloc[-1, i] * 100
    delta = (df.iloc[-1, i] - df.iloc[i, i]) * 100 if i < num_tasks - 1 else 0
    marker = '↑' if delta > 0 else '↓' if delta < 0 else '→'
    print(f"  {name}: {acc:.2f}% ({marker}{abs(delta):.2f}%)")

## 9. (Optional) Standalone EM Evaluation

Re-run generative exact-match evaluation on specific tasks with more samples or higher token limits.

In [ ]:
# Uncomment to re-evaluate with different settings:
# !python evaluate.py \
#     --weights {OUTPUT_PREFIX}_weights.pt \
#     --model_id ./abliterated \
#     --fast \
#     --max_samples 25 \
#     --max_new_tokens 512

## 10. Download Trained Weights & Results

Download the CASCADES adapter weights (.pt) and accuracy matrix (.csv) to your local machine.

In [ ]:
from google.colab import files

print("Downloading adapter weights...")
files.download(f"{OUTPUT_PREFIX}_weights.pt")

print("Downloading results CSV...")
files.download(f"{OUTPUT_PREFIX}_results.csv")

print("\n✓ Files downloaded. To load weights locally:")
print(f"  adapter_state = torch.load('{OUTPUT_PREFIX}_weights.pt')")

## 11. (Optional) Save to Google Drive

Persist weights to Drive so you don't lose them if the Colab runtime disconnects.

In [ ]:
import shutil, os

drive_dir = "/content/drive/MyDrive/cascades_runs"
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(f"{OUTPUT_PREFIX}_weights.pt", drive_dir)
shutil.copy(f"{OUTPUT_PREFIX}_results.csv", drive_dir)

print(f"✓ Saved to {drive_dir}/")
print(f"  {OUTPUT_PREFIX}_weights.pt")
print(f"  {OUTPUT_PREFIX}_results.csv")